In [11]:
import docs
from minsearch import Index
from typing import Any, Dict, List, TypedDict, Optional
from pydantic_ai import Agent

In [ ]:
# fetch and chunk github docs data
github_data = docs.read_github_data()
parsed_data = docs.parse_data(github_data)
chunks = docs.chunk_documents(parsed_data)

In [4]:
# create chunk index
index = Index(
    text_fields=["content", "filename", "title", "description"],
)

index.fit(chunks)

In [6]:
class SearchResult(TypedDict):
    """Represents a single search result entry."""
    start: int
    content: str
    title: str
    description: str
    filename: str


def search(query: str) -> List[SearchResult]:
    """
    Search the index for documents matching the given query.

    Args:
        query (str): The search query string.

    Returns:
        List[SearchResult]: A list of search results. Each result dictionary contains:
            - start (int): The starting position or offset within the source file.
            - content (str): A text excerpt or snippet containing the match.
            - filename (str): The path or name of the source file.
    """
    return index.search(
        query=query,
        num_results=5,
    )


### File Reading Capability

In [7]:
file_index = {}

for doc in parsed_data:
    filename = doc['filename']
    file_index[filename] = doc

In [9]:
def read_file(filename: str) -> Optional[str]:
    """
    Retrieve the content of a file from the repository.

    Args:
        filename (str): The name or path of the file to read.

    Returns:
        Optional[str]: The file content as a string if the file exists;
        otherwise, returns None.
    """
    if filename in file_index:
        return file_index[filename]['content']
    return None

### Agent Configuration

In [13]:
instructions = """
    You are an assistant that helps improve and generate high-quality documentation for the project.

    You have access to the following tools:
    - search — Use this to explore topics in depth. Make multiple search calls if needed to gather comprehensive information.
    - read_file — Use this when code snippets are missing or when you need to retrieve the full content of a file for context.

    Critical Rule

    Before generating or finalizing any code example or technical explanation, you must always call `read_file`
    to cross-check the correctness of the code.
    Do not rely solely on search results or assumptions — always verify by reading the actual file content.

    If `read_file` cannot be used or the file content is unavailable, clearly state:
    > "Unable to verify with read_file."

    When answering a question:
    1. Provide file references for all source materials.  
    Use this format:  
    [{filename}](https://github.com/evidentlyai/docs/blob/main/{filename})
    2. If the topic is covered in multiple documents, cite all relevant sources.
    3. Include code examples whenever they clarify or demonstrate the concept.
    4. Be concise, accurate, and helpful — focus on clarity and usability for developers.
    5. If documentation is missing or unclear, infer from context and note that explicitly.

    Example Citation

    See the full implementation in [metrics/api_reference.md](https://github.com/evidentlyai/docs/blob/main/metrics/api_reference.md).
""".strip()

In [14]:
agent_tools = [search, read_file]

In [15]:
agent = Agent(
    name="docs_agent",
    instructions=instructions,
    tools=agent_tools,
    model='gpt-4o-mini'
)

In [16]:
results = await agent.run(user_prompt='how do I run llm as a judge evals?')

In [17]:
print(results.output)

To run an LLM as a judge evaluator, you will use the `Evidently` library to create and evaluate responses against specified criteria. This process involves several steps, including creating an evaluation dataset, configuring your LLM as a judge, scoring responses, and evaluating the quality of the LLM itself.

### Step-by-Step Guide

1. **Install Evidently Library**:
   First, ensure you have the Evidently library installed. Use the following command:
   ```bash
   pip install evidently
   ```

2. **Import Required Modules**:
   You will need to import several modules to facilitate the evaluation process:
   ```python
   import pandas as pd
   import numpy as np
   from evidently import Dataset, DataDefinition, Report, BinaryClassification
   from evidently.presets import TextEvals
   from evidently.llm.templates import BinaryClassificationPromptTemplate
   import os
   ```

3. **Set Up OpenAI API Key**:
   Pass your OpenAI API key as an environment variable:
   ```python
   os.environ

In [18]:
messages = results.all_messages()

In [19]:
messages

[ModelRequest(parts=[UserPromptPart(content='how do I run llm as a judge evals?', timestamp=datetime.datetime(2025, 11, 10, 11, 41, 16, 492038, tzinfo=datetime.timezone.utc))], instructions='You are an assistant that helps improve and generate high-quality documentation for the project.\n\n    You have access to the following tools:\n    - search — Use this to explore topics in depth. Make multiple search calls if needed to gather comprehensive information.\n    - read_file — Use this when code snippets are missing or when you need to retrieve the full content of a file for context.\n\n    Critical Rule\n\n    Before generating or finalizing any code example or technical explanation, you must always call `read_file`\n    to cross-check the correctness of the code.\n    Do not rely solely on search results or assumptions — always verify by reading the actual file content.\n\n    If `read_file` cannot be used or the file content is unavailable, clearly state:\n    > "Unable to verify wit